# Lesson 2: Tool Calling & Structured Output

## What You'll Learn

1. **What are tools?** — How LLMs decide to call functions
2. **`@agent.tool` decorator** — Give your agent abilities
3. **DuckDB integration** — SQL queries on DataFrames
4. **Tool retry & error handling** — When tools fail, agents adapt
5. **Production patterns** — Input validation, timeouts, structured errors

---

## The Big Idea: Tools Turn an LLM into an Agent

In Lesson 1, our agent could only *think*. It had no way to interact with the world.

**Tools** are functions that the agent can *choose* to call during its reasoning.
The LLM doesn't execute the function itself — it says "I want to call `describe_dataset` with these arguments",
PydanticAI executes the function, and feeds the result back to the LLM.

```
User: "What's the average revenue per region?"

LLM thinks: I need to query the data. Let me call the SQL tool.
  → Calls: run_sql("SELECT region, AVG(revenue) FROM sales GROUP BY region")
  ← Gets: [{North: 3421.50}, {South: 3892.10}, ...]

LLM thinks: Now I have the data. I can answer the question.
  → Returns: AnalysisResult(answer="The South region has the highest...")
```

### How does the LLM "decide" to call a tool?

PydanticAI sends the LLM a **tool schema** — a JSON description of each available function:
- Function name
- Description (from docstring)
- Parameter names, types, and descriptions

The LLM is trained to recognize when it should call a tool vs. respond directly.
This is called **function calling** (OpenAI) or **tool use** (Anthropic).

---

## Setup

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# Resolve project root robustly whether launched from repo root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / 'src').exists() and (cwd / 'data').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').exists() and (cwd.parent / 'data').exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_DIR = PROJECT_ROOT / 'data'

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"

import pandas as pd
import duckdb
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelRetry

# Load our sample data
sales_df = pd.read_csv(DATA_DIR / 'sample_sales.csv')
employees_df = pd.read_csv(DATA_DIR / 'sample_employees.csv')

print(f"Project root: {PROJECT_ROOT}")
print(f"Sales data: {sales_df.shape}")
print(f"Employee data: {employees_df.shape}")


Sales data: (40, 8)
Employee data: (20, 8)


In [2]:
from pydantic_ai.exceptions import ModelHTTPError


def run_agent_checked(agent, *args, **kwargs):
    """Run agent safely and surface LM Studio template failures clearly."""
    try:
        return agent.run_sync(*args, **kwargs)
    except ModelHTTPError as exc:
        message = str(exc)
        if "No user query found in messages" in message:
            raise RuntimeError(
                "LM Studio prompt template is incompatible with this notebook flow.\n"
                "Fix in LM Studio:\n"
                "1) Use an lmstudio-community model/template for your model family.\n"
                "2) In My Models -> Prompt Template, remove logic that raises when no user query is found.\n"
                "3) Disable model thinking/reasoning output for API calls with strict structured outputs.\n"
                "4) Reload the model and rerun from the setup cell."
            ) from exc
        raise


---

## Part 1: Your First Tool — `@agent.tool_plain`

The simplest kind of tool. It doesn't need access to agent dependencies — just takes arguments and returns a result.

### `@agent.tool_plain` vs `@agent.tool` — When to use which

| Decorator | Access to Dependencies | Use When |
|-----------|----------------------|----------|
| `@agent.tool_plain` | No `RunContext` | Tool is self-contained (e.g., math, string formatting, static data) |
| `@agent.tool` | Full `RunContext[Deps]` | Tool needs runtime data (DataFrames, DB connections, configs) |

**Rule of thumb**: Start with `@agent.tool_plain` for simple, standalone functions.
Switch to `@agent.tool` when the function needs anything that's passed via `deps=`.

### How does the LLM "choose" to call a tool?

PydanticAI converts each tool into a **JSON schema** that the LLM sees:
```json
{
  "name": "describe_sales_data",
  "description": "Get a description of the sales dataset including columns, shape, and basic stats.",
  "parameters": {}
}
```
The LLM is trained to recognize when a tool is helpful and emit a special "tool call" token
instead of a regular text response. The **docstring** is critical — it's the tool's description
that the LLM reads to decide whether to call it.

In [3]:
# -----------------------------------------------------------------
# A simple agent with one tool: describe a dataset.
# The LLM can choose to call this tool when it needs data info.
# -----------------------------------------------------------------

data_agent = Agent(
    "openai:local-model",
    system_prompt=(
        "You are a data analyst. You have access to tools that let you "
        "inspect and query datasets. Always use tools before answering "
        "data questions — never guess."
    ),
)


@data_agent.tool_plain  # <-- no RunContext needed
def describe_sales_data() -> str:
    """Get a description of the sales dataset including columns, shape, and basic stats."""
    info_lines = [
        f"Shape: {sales_df.shape[0]} rows x {sales_df.shape[1]} columns",
        f"Columns: {', '.join(sales_df.columns)}",
        f"\nFirst 5 rows:\n{sales_df.head().to_string()}",
        f"\nBasic stats:\n{sales_df.describe().to_string()}",
    ]
    return "\n".join(info_lines)


result = run_agent_checked(data_agent, "What columns are in the sales dataset and how many rows?")
print(result.output)
print(f"\n--- Tokens used: {result.usage().input_tokens + result.usage().output_tokens} ---")



The sales dataset contains **8 columns** and **40 rows**.

Here are the column names:
1.  date
2.  product
3.  category
4.  region
5.  quantity
6.  unit_price
7.  revenue
8.  cost

--- Tokens used: 1463 ---


In [4]:
# -----------------------------------------------------------------
# Let's trace what happened — see the tool call in the message history.
# This is crucial for debugging agents.
# -----------------------------------------------------------------

from rich import print as rprint

for msg in result.all_messages():
    rprint(f"[bold]{msg.kind}[/bold]")
    for part in msg.parts:
        part_kind = getattr(part, 'part_kind', 'unknown')
        if part_kind == 'tool-call':
            rprint(f"  Tool call: [green]{part.tool_name}[/green]({part.args})")
        elif part_kind == 'tool-return':
            content_preview = str(part.content)[:200]
            rprint(f"  Tool return: [blue]{content_preview}...[/blue]")
        elif part_kind == 'text':
            rprint(f"  Text: {str(part.content)[:200]}...")
        else:
            rprint(f"  {part_kind}: {str(part)[:100]}")
    print()

request

system-prompt: SystemPromptPart(content='You are a data analyst. You have access to tools that let you inspect 
and

user-prompt: UserPromptPart(content='What columns are in the sales dataset and how many rows?', timestamp=datetim

response

thinking: ThinkingPart(content='\n\n', id='content', provider_name='openai')

...

request

Tool return: Shape: 40 rows x 8 columns
Columns: date, product, category, region, quantity, unit_price, revenue, cost

First 5 rows:
         date   product     category region  quantity  unit_price  revenue     c...

response

thinking: ThinkingPart(content='\n\n', id='content', provider_name='openai')

Text: 

The sales dataset contains **8 columns** and **40 rows**.

Here are the column names:
1.  date
2.  product
3.  category
4.  region
5.  quantity
6.  unit_price
7.  revenue
8.  cost...

### Message flow breakdown — Why are there 2 API calls?

```
1. request  → [system-prompt] + [user-prompt: "What columns..."]
2. response → [tool-call: describe_sales_data()]           ← API call #1
3. request  → [tool-return: "Shape: 40 rows x 8 columns..."]
4. response → [text: "The sales dataset has 40 rows..."]   ← API call #2
```

**Why 2 calls?** The LLM can only do one thing per response: either generate text OR request a tool call.
It can't call a tool AND answer the question in the same response.

**Call #1**: LLM sees the question, decides it needs data, emits a tool-call request.
PydanticAI intercepts this, executes the function, and gets the result.

**Call #2**: PydanticAI sends the tool result back to the LLM. Now the LLM has the data
it needs and generates the final text answer.

This is the **tool calling loop**: the LLM keeps requesting tools until it has enough
information to answer, then it returns text. Each tool call = 1 extra API call.

### Cost implication

More tools = potentially more API calls. If the LLM calls 3 tools sequentially,
that's 4 API calls total (3 tool calls + 1 final answer). This is why tool call
budgets matter in production (covered in Lesson 4).

---

## Part 2: Tools with Dependencies — `@agent.tool`

Real tools need access to runtime data (the DataFrame, a database connection, etc.).
The `@agent.tool` decorator gives you `RunContext` — the same dependency injection from Lesson 1.

In [5]:
# -----------------------------------------------------------------
# Agent with DuckDB SQL tool — the agent can write SQL queries
# and run them against any DataFrame.
# -----------------------------------------------------------------

@dataclass
class DataDeps:
    """Runtime dependencies — available to all tools and prompts."""
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)


class SQLResult(BaseModel):
    """Structured output for SQL-based analysis."""
    answer: str = Field(description="Natural language answer")
    sql_query: str = Field(description="The SQL query used")
    confidence: float = Field(ge=0.0, le=1.0)


sql_agent = Agent(
    "openai:local-model",
    deps_type=DataDeps,
    output_type=SQLResult,
    system_prompt=(
        "You are a data analyst with SQL expertise. "
        "Use the available tools to inspect and query data. "
        "Always inspect the schema first, then write SQL."
    ),
)


@sql_agent.system_prompt
def add_table_info(ctx: RunContext[DataDeps]) -> str:
    """Tell the LLM what tables are available."""
    table_info = []
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        table_info.append(f"Table '{name}': {df.shape[0]} rows — Columns: {cols}")
    return "\nAvailable tables:\n" + "\n".join(table_info)


@sql_agent.tool
def run_sql(ctx: RunContext[DataDeps], query: str) -> str:
    """Execute a SQL query against the available tables. Use standard SQL syntax.
    Table names match the keys in the available tables list.
    Returns the query results as a formatted string.
    """
    conn = duckdb.connect()
    try:
        # Register all DataFrames as DuckDB tables
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)

        result = conn.execute(query).fetchdf()

        if len(result) == 0:
            return "Query returned 0 rows."

        # Limit output size to avoid context overflow
        if len(result) > 50:
            return (
                f"Query returned {len(result)} rows. Showing first 50:\n"
                f"{result.head(50).to_string()}"
            )
        return result.to_string()
    except Exception as e:
        return f"SQL Error: {e}"
    finally:
        conn.close()


@sql_agent.tool
def inspect_table(ctx: RunContext[DataDeps], table_name: str) -> str:
    """Get detailed schema info for a specific table.
    Returns column names, types, null counts, and sample values.
    """
    if table_name not in ctx.deps.tables:
        available = list(ctx.deps.tables.keys())
        return f"Table '{table_name}' not found. Available: {available}"

    from analyst.tools.schema_inspector import inspect_schema
    return inspect_schema(ctx.deps.tables[table_name], table_name)


# Run it!
deps = DataDeps(tables={"sales": sales_df, "employees": employees_df})

result = run_agent_checked(sql_agent, 
    "What is the total revenue by region? Which region is most profitable?",
    deps=deps,
)

print(f"Answer: {result.output.answer}")
print(f"\nSQL: {result.output.sql_query}")
print(f"Confidence: {result.output.confidence:.0%}")

Traceback (most recent call last):
  File "/Users/ayushkumar/Documents/python_stuff/learning/agentic-ai/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/zv/hn7c_thn009b17n8mp1gghyh0000gn/T/ipykernel_12507/1168927661.py", line 87, in <module>
    result = run_agent_checked(sql_agent,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/zv/hn7c_thn009b17n8mp1gghyh0000gn/T/ipykernel_12507/3983754553.py", line 7, in run_agent_checked
    return agent.run_sync(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ayushkumar/Documents/python_stuff/learning/agentic-ai/.venv/lib/python3.11/site-packages/pydantic_ai/agent/abstract.py", line 386, in run_sync
    return _utils.get_event_loop().run_until_complete(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ayushkumar/Documents/python_stuff/learning/agentic-ai/.venv/lib/pytho

In [ ]:
# Let's ask a cross-table question
result2 = run_agent_checked(sql_agent, 
    "What is the average salary in the Engineering department, "
    "and how does it compare to the company average?",
    deps=deps,
)

print(f"Answer: {result2.output.answer}")
print(f"SQL: {result2.output.sql_query}")
print(f"Confidence: {result2.output.confidence:.0%}")

---

## Part 3: Tool Error Handling & Retry

### The Problem
LLMs write bad SQL sometimes. Column names are wrong, syntax is off, or the query logic is flawed.
A production agent must handle this gracefully — not crash.

### The Solution: `ModelRetry`
When a tool detects an error, it raises `ModelRetry(error_message)`. This:
1. Sends the error message back to the LLM as a tool return
2. The LLM sees its mistake and generates a corrected tool call
3. The corrected tool call is executed
4. This repeats up to `retries` times

### What if it always fails?

If the LLM can't produce valid output after `retries` attempts, PydanticAI raises
a `UnexpectedModelBehavior` exception. Your code should catch this:

```python
try:
    result = agent.run_sync("complex question", deps=deps)
except UnexpectedModelBehavior:
    # Fall back to a simpler approach or ask the user to rephrase
    result = "Sorry, I couldn't answer that question."
```

### Self-correction in practice

This is one of the most powerful patterns in agentic AI. The LLM learns from its
own mistakes *within a single query*:
- Wrong column name → error tells it the right columns → fixes the query
- Syntax error → error shows the exact issue → corrects syntax
- Wrong table → error shows available tables → uses correct table

In [ ]:
# -----------------------------------------------------------------
# Agent with self-correcting SQL tool.
# If the SQL fails, the error goes back to the LLM for retry.
# -----------------------------------------------------------------

robust_agent = Agent(
    "openai:local-model",
    deps_type=DataDeps,
    output_type=SQLResult,
    system_prompt=(
        "You are a data analyst. Use SQL to answer questions. "
        "If a query fails, read the error carefully and fix your SQL."
    ),
    retries=3,  # <-- max retries for tool errors
)


@robust_agent.system_prompt
def add_tables(ctx: RunContext[DataDeps]) -> str:
    table_info = []
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        table_info.append(f"Table '{name}': {df.shape[0]} rows — Columns: {cols}")
    return "\nAvailable tables:\n" + "\n".join(table_info)


@robust_agent.tool
def run_sql_safe(ctx: RunContext[DataDeps], query: str) -> str:
    """Execute SQL against available tables. Fix and retry if the query fails."""
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)

        result = conn.execute(query).fetchdf()

        if len(result) > 50:
            return f"Showing first 50 of {len(result)} rows:\n{result.head(50).to_string()}"
        return result.to_string()

    except Exception as e:
        # ModelRetry sends the error back to the LLM to self-correct
        raise ModelRetry(
            f"SQL execution failed. Fix your query and try again.\n"
            f"Error: {e}\n"
            f"Your query was: {query}"
        )
    finally:
        conn.close()


# Intentionally tricky question — might require self-correction
result = run_agent_checked(robust_agent, 
    "Calculate the profit margin (revenue - cost) / revenue for each product. "
    "Show as a percentage, sorted highest to lowest.",
    deps=deps,
)

print(f"Answer: {result.output.answer}")
print(f"\nSQL: {result.output.sql_query}")
print(f"\nAPI requests made: {result.usage().requests} (includes retries)")

### How retry works under the hood

```
1. LLM generates SQL: SELECT profit_margin FROM sales  ← wrong, column doesn't exist
2. Tool executes → DuckDB error: "column profit_margin not found"
3. ModelRetry sends error back to LLM
4. LLM generates new SQL: SELECT (revenue - cost) / revenue * 100 AS profit_margin FROM sales
5. Tool executes → Success! Returns results.
6. LLM formulates answer from results.
```

The `retries=3` parameter limits this loop to prevent infinite retry cycles.

---

## Part 4: Multiple Tools — Building a Toolkit

Real agents have multiple tools. The LLM chooses which one to call based on the task.
Let's give our agent a full analytics toolkit.

In [ ]:
# -----------------------------------------------------------------
# Multi-tool agent: SQL + schema inspection + value counts
# The LLM picks the right tool for each sub-task.
# -----------------------------------------------------------------

class DetailedAnalysis(BaseModel):
    """A thorough data analysis result."""
    answer: str = Field(description="Complete answer to the question")
    method: str = Field(description="What approach/tools were used")
    key_findings: list[str] = Field(description="Bullet-point findings")
    sql_queries_used: list[str] = Field(description="SQL queries executed")
    confidence: float = Field(ge=0.0, le=1.0)


full_agent = Agent(
    "openai:local-model",
    deps_type=DataDeps,
    output_type=DetailedAnalysis,
    system_prompt=(
        "You are an expert data analyst with access to multiple tools.\n"
        "Strategy:\n"
        "1. First inspect the table schema to understand the data\n"
        "2. Then run SQL queries to answer the question\n"
        "3. Use value counts for categorical breakdowns\n"
        "4. Provide a thorough answer with key findings"
    ),
    retries=3,
)


@full_agent.system_prompt
def tables_context(ctx: RunContext[DataDeps]) -> str:
    info = []
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        info.append(f"Table '{name}': {df.shape[0]} rows — {cols}")
    return "\nAvailable tables:\n" + "\n".join(info)


@full_agent.tool
def query_sql(ctx: RunContext[DataDeps], sql: str) -> str:
    """Run a SQL query against available tables. Returns results as text."""
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(sql).fetchdf()
        if len(result) > 50:
            return f"First 50 of {len(result)} rows:\n{result.head(50).to_string()}"
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}\nQuery: {sql}")
    finally:
        conn.close()


@full_agent.tool
def get_schema(ctx: RunContext[DataDeps], table_name: str) -> str:
    """Get detailed schema for a table: column names, types, samples, nulls."""
    if table_name not in ctx.deps.tables:
        return f"Table not found. Available: {list(ctx.deps.tables.keys())}"
    from analyst.tools.schema_inspector import inspect_schema
    return inspect_schema(ctx.deps.tables[table_name], table_name)


@full_agent.tool
def get_value_counts(ctx: RunContext[DataDeps], table_name: str, column: str) -> str:
    """Get value counts for a categorical column. Shows frequency of each unique value."""
    if table_name not in ctx.deps.tables:
        return f"Table not found. Available: {list(ctx.deps.tables.keys())}"
    from analyst.tools.schema_inspector import inspect_value_counts
    return inspect_value_counts(ctx.deps.tables[table_name], column)


# Complex question requiring multiple tool calls
result = run_agent_checked(full_agent, 
    "Compare the Electronics vs Home category: which has higher total revenue, "
    "higher profit margins, and which products are the top sellers in each?",
    deps=deps,
)

analysis = result.output
print(f"Answer:\n{analysis.answer}\n")
print(f"Method: {analysis.method}")
print(f"\nKey findings:")
for f in analysis.key_findings:
    print(f"  - {f}")
print(f"\nSQL queries used:")
for q in analysis.sql_queries_used:
    print(f"  {q}")
print(f"\nConfidence: {analysis.confidence:.0%}")
print(f"Total requests: {result.usage().requests}")

---

## Part 5: Output Validators — Quality Gates

Structured output validates the *format* (is it valid JSON matching the schema?).
But what about *content quality*? (Does the answer actually contain useful numbers?)

**Output validators** run after the LLM produces output but before it's returned to you.
They can either:
- **Accept**: Return the output as-is
- **Reject**: Raise `ModelRetry` to send the output back for improvement

### The validation pipeline

```
LLM generates output
       |
       v
[Pydantic validation]  ← Format check (types, ranges, required fields)
       |
       v
[@output_validator]    ← Content check (has numbers? not too vague?)
       |
       v
   Return to user
```

### When output validators retry, what happens?

The LLM receives its own previous output + the validation error message.
It then generates a new output that attempts to fix the issue. This is
different from tool retries — here the *answer itself* is being improved.

**Important**: Output validators share the same `retries` budget as tool retries.
If `retries=3`, those 3 retries are shared across both tool errors AND output
validation failures. Set it high enough to account for both.

In [ ]:
# -----------------------------------------------------------------
# Agent with an output validator:
# If the answer is too vague or lacks numbers, retry.
# -----------------------------------------------------------------

validated_agent = Agent(
    "openai:local-model",
    deps_type=DataDeps,
    output_type=SQLResult,
    system_prompt=(
        "You are a precise data analyst. Always include specific numbers in your answers. "
        "Never say 'approximately' without giving the exact figure too."
    ),
    retries=3,
)


@validated_agent.system_prompt
def add_context(ctx: RunContext[DataDeps]) -> str:
    info = []
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        info.append(f"Table '{name}': {df.shape[0]} rows — {cols}")
    return "\nAvailable tables:\n" + "\n".join(info)


@validated_agent.tool
def execute_sql(ctx: RunContext[DataDeps], query: str) -> str:
    """Run SQL against available tables."""
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        return conn.execute(query).fetchdf().to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


@validated_agent.output_validator
def check_answer_quality(ctx: RunContext[DataDeps], output: SQLResult) -> SQLResult:
    """Validate that the answer contains specific numbers."""
    # Check if the answer contains at least one number
    has_numbers = any(char.isdigit() for char in output.answer)
    if not has_numbers:
        raise ModelRetry(
            "Your answer must include specific numbers from the data. "
            "Don't just describe trends — give exact figures."
        )

    # Check confidence isn't suspiciously low
    if output.confidence < 0.3:
        raise ModelRetry(
            "Your confidence is very low. Re-examine your SQL query and results. "
            "If the data supports the answer, increase confidence."
        )

    return output


result = run_agent_checked(validated_agent, 
    "What is the total revenue and which product generates the most?",
    deps=deps,
)

print(f"Answer: {result.output.answer}")
print(f"SQL: {result.output.sql_query}")
print(f"Confidence: {result.output.confidence:.0%}")
print(f"Requests: {result.usage().requests} (retries show as extra requests)")

---

## Summary & Key Takeaways

### What we built
| Component | Pattern | Why |
|-----------|---------|-----|
| `@agent.tool_plain` | Simple tools (no deps) | Quick functions the LLM can call |
| `@agent.tool` | Tools with `RunContext` | Access runtime data (DataFrames, configs) |
| DuckDB integration | SQL on DataFrames | Let the agent write real queries |
| `ModelRetry` | Self-correcting tools | Agent fixes its own SQL errors |
| Multi-tool agent | Pick the right tool | LLM chooses between schema/SQL/value counts |
| Output validator | Quality gates | Ensure answer quality before returning |

### Production patterns introduced
1. **Tool error → `ModelRetry`** — Never crash. Give the LLM a chance to fix it.
2. **`retries=3`** — Always set a retry limit to prevent infinite loops.
3. **Output size limits** — Truncate large results to avoid context overflow.
4. **Output validation** — Check content quality, not just format.
5. **Message tracing** — Log every tool call for debugging.

### What's next
- **Lesson 3**: The agent writes *Python code* (not just SQL) and runs it in a sandbox
- **Lesson 4**: Multi-step reasoning — the agent plans, executes, and iterates

---

## Exercises

1. **Add a `compute_correlation` tool** — Takes two column names, returns Pearson correlation with interpretation.

2. **Cross-table queries** — Ask the agent a question that requires joining sales and employees data (e.g., "What if each region had a sales team — estimate team size from employee data").

3. **Build a tool usage tracker** — Count how many times each tool is called across queries. Which tools does the LLM prefer?

4. **Try the same queries on Claude** — Change `openai:local-model` to `anthropic:claude-3-5-haiku-latest`. Does it generate different SQL? Which is more accurate?